# Where and what to buy

**Brief: Where (neighbourhoods) + What (property types and mix).**

- Revenue, nightly price and occupancy by neighbourhood and type
- Which segments have the best yield once purchase price is included
- Watch out: segments with few listings (`n`) have unreliable medians

In [1]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `seg`: neighbourhood × type × bedrooms (active entire homes)
- `uw`: segments + purchase price, costs, NOI, net yield

### From 01 (keep in mind)
- **Occupancy & revenue** = Inside Airbnb's estimates (`estimated_occupancy_l365d`, `estimated_revenue_l365d`), 12 months to 15 June 2026. Estimates, not real bookings.
- **Price** = one night's quote around June–July 2026 (World Cup period), so revenue may be a bit high.
- **`is_active`** = reviewed in the last 12 months. Only active listings are used for segments.
- Cleaned data: 6,509 → 4,447 listings (no price, price outside $30–$2,000, 90+ night minimum removed: BC/Vancouver short-term = under 90 nights).

In [2]:
seg = pd.read_csv(T / "segments.csv", dtype={"bedrooms_grp": str})
uw = pd.read_csv(T / "segments_underwritten.csv", dtype={"bedrooms_grp": str})

## Quick look

In [3]:
uw[uw["n"] >= 5].sort_values("net_yield", ascending=False).head(20)

,neighbourhood_cleansed,bucket,bedrooms_grp,n,adr_median,occ_median,rev_median,rev_p25,rev_p75,rating,share_commercial,share_licensed_est,prop_type,median_value,price_cad,noi,cash_in,net_yield
14,Downtown Eastside,condo,3+,13,491.00,0.33,"81,345.00","20,790.00","117,612.00",4.67,0.46,0.85,condo,"617,500.00","926,250.00","43,849.30","988,300.00",0.04
12,Downtown Eastside,condo,1,152,368.16,0.35,"49,140.00","17,515.50","75,666.75",4.88,0.14,0.95,condo,"617,500.00","524,875.00","22,831.97","570,870.00",0.04
6,Downtown,condo,1,374,409.85,0.35,"51,788.00","21,264.75","83,044.50",4.88,0.15,0.95,condo,"758,200.00","644,470.00","24,300.31","695,248.80",0.03
7,Downtown,condo,2,292,542.12,0.33,"63,091.50","24,832.00","101,596.50",4.88,0.18,0.98,condo,"758,200.00","871,930.00","31,417.34","931,807.20",0.03
8,Downtown,condo,3+,121,575.00,0.33,"75,345.00","28,758.00","125,715.00",4.82,0.38,0.93,condo,"758,200.00","1,137,300.00","39,076.15","1,207,792.00",0.03
109,West End,condo,2,25,513.53,0.28,"59,953.00","27,510.00","91,837.00",4.91,0.16,0.96,condo,"779,636.00","896,581.40","29,177.83","957,444.66",0.03
54,Kitsilano,condo,1,33,370.88,0.31,"46,221.00","14,858.00","68,544.00",4.88,0.12,1.00,condo,"988,900.00","840,565.00","19,870.79","899,187.60",0.02
13,Downtown Eastside,condo,2,84,542.28,0.23,"41,124.00","16,985.25","77,004.00",4.88,0.25,0.93,condo,"617,500.00","710,125.00","16,745.18","763,530.00",0.02
83,Riley Park,condo,2,7,353.20,0.31,"65,695.00","26,764.00","77,775.00",4.88,0.00,1.00,condo,"1,194,000.00","1,373,100.00","31,710.25","1,453,024.00",0.02
23,Fairview,condo,1,5,370.00,0.23,"39,045.00","17,760.00","43,041.00",4.85,0.00,1.00,condo,"878,000.00","746,300.00","15,202.15","801,152.00",0.02


In [4]:
uw.groupby("bucket")[["rev_median", "price_cad", "net_yield"]].median()

,rev_median,price_cad,net_yield
bucket,,,
condo,"31,797.00","1,079,850.00",0.01
house,"35,710.50","2,601,075.00",0.01


## Analysis
_Start here._

### Step 1: Demand by neighbourhood
Active entire-home listings only (what DDD would run). Typical (median) value per neighbourhood. Hover a bar to see all the numbers for that neighbourhood.

In [5]:
import altair as alt

df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)
d = df[df["is_active"] & (df["room_type"] == "Entire home/apt")]

hood = (d.groupby("neighbourhood_cleansed")
          .agg(listings=("id", "size"), revenue=("revenue", "median"), total_revenue=("revenue", "sum"),
               nights_booked=("estimated_occupancy_l365d", "median"), price=("price_num", "median"))
          .reset_index())
hood

,neighbourhood_cleansed,listings,revenue,total_revenue,nights_booked,price
0,Arbutus Ridge,64,"38,682.00","3,579,934.00",108.00,400.00
1,Downtown,798,"58,486.00","51,770,098.00",126.00,460.97
2,Downtown Eastside,290,"44,043.00","15,640,536.00",102.00,401.18
3,Dunbar Southlands,125,"32,886.00","5,417,985.00",66.00,385.72
4,Fairview,18,"45,239.00","1,114,288.00",96.00,435.75
5,Grandview-Woodland,128,"36,199.50","5,402,280.00",93.00,350.88
6,Hastings-Sunrise,176,"42,266.50","8,842,193.00",123.00,312.88
7,Kensington-Cedar Cottage,217,"40,741.00","10,291,055.00",120.00,350.00
8,Kerrisdale,46,"32,396.50","2,093,233.00",72.00,399.98
9,Killarney,73,"24,960.00","2,864,865.00",78.00,319.53


In [6]:
def bar(col, title, fmt, note=""):
    tip = [alt.Tooltip("neighbourhood_cleansed:N", title="Neighbourhood"),
           alt.Tooltip("revenue:Q", title="Typical yearly revenue", format="$,.0f"),
           alt.Tooltip("nights_booked:Q", title="Typical nights booked / year", format=",.0f"),
           alt.Tooltip("price:Q", title="Typical nightly price", format="$,.0f"),
           alt.Tooltip("listings:Q", title="Listings")]
    chart = (alt.Chart(hood)
             .mark_bar(color="#2a78d6", cornerRadiusEnd=4, height=16)
             .encode(x=alt.X(f"{col}:Q", title=None, axis=alt.Axis(format=fmt, grid=True, gridColor="#eee")),
                     y=alt.Y("neighbourhood_cleansed:N", sort="-x", title=None),
                     tooltip=tip)
             .properties(width=550, height=alt.Step(20),
                         title=alt.Title(title, subtitle=note, anchor="start", subtitleColor="#52514e"))
             .configure_view(stroke=None))
    display(chart)

bar("revenue", "Typical yearly revenue per listing (CAD)", "$,.0f",
    note="Yearly revenue = nights booked × nightly price · 12 months to June 2026")
bar("nights_booked", "Typical occupancy per year", ",.0f",
    note="Occupancy = nights booked out of 365 · 12 months to June 2026")
bar("listings", "Number of active whole-home listings", ",d",
    note="Supply: more listings = proven demand, but also more competition")
bar("total_revenue", "Total yearly revenue, all active whole-home listings (CAD)", "$,.0f",
    note="Size of the market = sum of every listing's revenue · 12 months to June 2026")
bar("price", "Typical nightly price (CAD)", "$,.0f",
    note="Quote for one night around June–July 2026 (World Cup period)")

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### Step 2: Return per neighbourhood → top 5
**Gross yield = estimated revenue ÷ purchase price**
- Revenue = nights booked × nightly price (Inside Airbnb estimate, 12 months to June 2026)
- Purchase price = median 2026 assessed value for that neighbourhood × type (City of Vancouver property tax report), condos scaled by bedrooms

Operating costs are left out here: they shrink every neighbourhood about equally, so the ranking doesn't change (tested: same top 5, rank agreement 0.99). They come back only for the final 2027 profit number.

Filter 1: at least 50 listings (reliable numbers). Filter 2: rank by typical gross yield. Suites excluded (can't buy without the house). Strathcona has no homes in the tax data, so it drops out.

In [7]:
from src import portfolio

d = d.copy()
d["bedrooms_grp"] = d["bedrooms"].fillna(1).clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
d_ret = portfolio.add_purchase_prices(d)                      # one row per listing, adds price_cad
d_ret["gross_yield"] = d_ret["revenue"] / d_ret["price_cad"]

ret = (d_ret.groupby("neighbourhood_cleansed")
            .agg(listings=("id", "size"), revenue=("revenue", "median"),
                 purchase_price=("price_cad", "median"), gross_yield=("gross_yield", "median"))
            .reset_index())
shortlist = ret[ret["listings"] >= 50].sort_values("gross_yield", ascending=False)
top5 = shortlist.head(5)
shortlist

,neighbourhood_cleansed,listings,revenue,purchase_price,gross_yield
1,Downtown,796,"58,486.00","871,930.00",0.07
2,Downtown Eastside,279,"44,472.00","524,875.00",0.07
20,West End,78,"41,136.00","662,690.60",0.05
12,Mount Pleasant,109,"33,886.00","859,165.00",0.04
7,Kensington-Cedar Cottage,137,"41,904.00","1,824,000.00",0.02
6,Hastings-Sunrise,112,"35,819.50","1,739,050.00",0.02
14,Renfrew-Collingwood,102,"33,180.00","1,786,000.00",0.02
10,Kitsilano,167,"43,368.00","2,761,500.00",0.02
5,Grandview-Woodland,87,"25,884.00","1,843,000.00",0.02
19,Victoria-Fraserview,93,"33,480.00","2,025,000.00",0.02


### Step 2b: Fair comparison — condos vs condos, houses vs houses
Step 2 mixes types: Downtown is mostly condos (cheap to buy), other areas mostly houses (expensive), so condos win by default. 
Here each type is ranked on its own. Same filter: at least 50 listings of that type.

In [8]:
by_type = (d_ret.groupby(["bucket", "neighbourhood_cleansed"])
                .agg(listings=("id", "size"), revenue=("revenue", "median"),
                     purchase_price=("price_cad", "median"), gross_yield=("gross_yield", "median"))
                .reset_index())
by_type = by_type[by_type["listings"] >= 50].sort_values(["bucket", "gross_yield"], ascending=[True, False])

for bucket, t in by_type.groupby("bucket"):
    print(f"\n=== {bucket.upper()} ===")
    print(t.drop(columns="bucket").to_string(index=False, formatters={
        "revenue": "${:,.0f}".format, "purchase_price": "${:,.0f}".format,
        "gross_yield": "{:.1%}".format}))


=== CONDO ===
neighbourhood_cleansed  listings revenue purchase_price gross_yield
     Downtown Eastside       249 $45,390       $524,875        7.6%
              Downtown       787 $58,644       $871,930        7.4%
              West End        74 $41,136       $662,691        5.8%
        Mount Pleasant        74 $29,886       $635,035        4.6%
             Kitsilano        51 $41,247       $840,565        3.6%

=== HOUSE ===
  neighbourhood_cleansed  listings revenue purchase_price gross_yield
Kensington-Cedar Cottage       127 $41,154     $1,824,000        2.3%
        Hastings-Sunrise       103 $35,999     $1,739,050        2.1%
     Renfrew-Collingwood        95 $36,378     $1,786,000        2.0%
      Grandview-Woodland        63 $33,696     $1,843,000        1.8%
               Kitsilano       116 $47,352     $2,761,500        1.7%
     Victoria-Fraserview        91 $33,480     $2,025,000        1.7%
                  Sunset        79 $27,936     $1,880,000        1.5%
  